In [1]:

# IMPORT LIBRARIES
import os
# Import random module for setting random seeds and randomized operations
import random
# Import NumPy library for numerical arrays, matrix math, and vector operations
import numpy as np
# Import pandas library for DataFrame handling and spreadsheet manipulation
import pandas as pd

# Import PIL Image to open, resize, and convert image files
from PIL import Image

# Import PyTorch core deep learning framework
import torch
# Import PyTorch core deep learning framework
import torch.nn as nn

# Import Dataset base class and DataLoader for batching and loading data
from torch.utils.data import Dataset, DataLoader

# Import torchvision pretrained models and image transform pipelines
from torchvision import models, transforms

# Import dataset splitting utilities from scikit-learn
from sklearn.model_selection import train_test_split

# Import evaluation metrics (accuracy, confusion matrix, precision, recall, F1, ROC-AUC)
from sklearn.metrics import (accuracy_score,confusion_matrix,classification_report,precision_score,recall_score,f1_score,roc_auc_score)

# Import matplotlib pyplot for creating charts and visual plots
import matplotlib.pyplot as plt

# RANDOM SEED

# Define fixed random seed value to ensure reproducible results across runs
SEED = 42

# Seed Python built-in random number generator
random.seed(SEED)
# Seed NumPy random number generator
np.random.seed(SEED)
# Seed PyTorch CPU random number generator
torch.manual_seed(SEED)

# Check if a CUDA-capable GPU is available
if torch.cuda.is_available():
    # Seed all GPU CUDA random number generators for reproducibility
    torch.cuda.manual_seed_all(SEED)

# PATHS

# Define base project directory path containing datasets and assets
path2data = (
    r"F:/Amrita_2024/DEEP LEARNING FOR SIGNAL & IMAGE _21AIE312/"
    r"Python_neural network/"
)

# Define directory path containing raw training image files
path2train = (
    r"F:/Amrita_2024/DEEP LEARNING FOR SIGNAL & IMAGE _21AIE312/"
    r"Python_neural network/training_drion/"
)

# Construct full file path to the Excel spreadsheet containing labels
path2labels = os.path.join(
    path2data,
    "drion_training.xlsx"
)


# READ EXCEL FILE

# Read the Excel spreadsheet into a pandas DataFrame
labels_df = pd.read_excel(path2labels)

print("=" * 70)
print("DRION DATASET INFORMATION")
print("=" * 70)

print("\nFirst five rows:")
print(labels_df.head())

print("\nDataset shape:")
print(labels_df.shape)

print("\nColumns:")
print(labels_df.columns.tolist())

print("\nClass distribution:")
print(labels_df["label"].value_counts())

DRION DATASET INFORMATION

First five rows:
   ID        imgName  label
0   1  image_001.jpg      0
1   2  image_002.jpg      0
2   3  image_003.jpg      0
3   4  image_004.jpg      0
4   5  image_005.jpg      0

Dataset shape:
(86, 3)

Columns:
['ID', 'imgName', 'label']

Class distribution:
label
0    47
1    39
Name: count, dtype: int64


In [ ]:

# GET IMAGE PATHS AND LABELS

# Initialize empty list to store valid image file paths
image_paths = []
# Initialize empty list to store ground truth labels
image_labels = []
# Initialize empty list to store image filenames
image_names = []

print("\n" + "=" * 70)
print("CHECKING IMAGE FILES")
print("=" * 70)


# Iterate over each row in the labels DataFrame
for _, row in labels_df.iterrows():

    # Extract image filename string from current DataFrame row
    filename = str(row["imgName"])
    # Extract label value from current DataFrame row
    label = row["label"]

    # Construct full file path for the current image
    image_path = os.path.join(
        path2train,
        filename
    )

    # Verify whether the image file actually exists on disk
    if os.path.exists(image_path):

        # Append valid image path to image_paths list
        image_paths.append(image_path)
        # Append corresponding label to image_labels list
        image_labels.append(label)
        # Append filename to image_names list
        image_names.append(filename)

    else:

        print(
            "Image not found:",
            image_path
        )


print("\nTotal valid images:", len(image_paths))

print(
    "Total missing images:",
    len(labels_df) - len(image_paths)
)

if len(image_paths) == 0:
    raise RuntimeError(
        "No images were found. Check path2train and imgName."
    )


CHECKING IMAGE FILES

Total valid images: 86
Total missing images: 0


In [ ]:

#  LABEL ENCODING

# Extract distinct unique class label values from the dataset
unique_labels = list(pd.unique(image_labels))

print("\n" + "=" * 70)
print("LABEL ENCODING")
print("=" * 70)

print("\nOriginal classes:")
print(unique_labels)

# Explicitly make Glaucoma = 1 and Normal = 0
# when those labels exist.

normalized_labels = {
    str(label).strip().lower(): label
    for label in unique_labels
}


if (
    "normal" in normalized_labels
    and "glaucoma" in normalized_labels
):

    normal_label = normalized_labels["normal"]
    glaucoma_label = normalized_labels["glaucoma"]

    # Create dictionary mapping class label strings to numeric indices (0, 1)
    label_to_index = {
        normal_label: 0,
        glaucoma_label: 1
    }

else:

    # Generic binary-label fallback
    sorted_labels = sorted(unique_labels)

    # Create dictionary mapping class label strings to numeric indices (0, 1)
    label_to_index = {
        label: index
        for index, label in enumerate(sorted_labels)
    }


# Create reverse dictionary mapping numeric indices back to class label strings
index_to_label = {
    index: label
    for label, index in label_to_index.items()
}


# Encode all label strings into a NumPy array of integer class indices
encoded_labels = np.array(
    [
        label_to_index[label]
        for label in image_labels
    ],
    dtype=np.int64
)


print("\nLabel mapping:")
print(label_to_index)

print("\nEncoded class distribution:")
print(
    pd.Series(encoded_labels).value_counts().sort_index()
)

if len(label_to_index) != 2:
    raise ValueError(
        "This AdaBoost code expects exactly two classes."
    )


LABEL ENCODING

Original classes:
[np.int64(0), np.int64(1)]

Label mapping:
{np.int64(0): 0, np.int64(1): 1}

Encoded class distribution:
0    47
1    39
Name: count, dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_16812\3119913429.py:5: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  unique_labels = list(pd.unique(image_labels))


In [ ]:

#  TRAIN / TEST SPLIT

# Split dataset into stratified 80% train and 20% test partitions
train_paths, test_paths, train_labels, test_labels = train_test_split(

    image_paths,

    encoded_labels,

    test_size=0.20,

    random_state=SEED,

    stratify=encoded_labels
)


# Convert labels to NumPy arrays
train_labels = np.asarray(
    train_labels,
    dtype=np.int64
)

test_labels = np.asarray(
    test_labels,
    dtype=np.int64
)


print("\n" + "=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

print(
    "Training images:",
    len(train_paths)
)

print(
    "Testing images :",
    len(test_paths)
)

print("\nTraining class distribution:")
print(
    pd.Series(train_labels).value_counts().sort_index()
)

print("\nTesting class distribution:")
print(
    pd.Series(test_labels).value_counts().sort_index()
)


TRAIN / TEST SPLIT
Training images: 68
Testing images : 18

Training class distribution:
0    37
1    31
Name: count, dtype: int64

Testing class distribution:
0    10
1     8
Name: count, dtype: int64


In [ ]:

#  DEVICE AND IMAGE TRANSFORMS


# Select GPU device ('cuda') if available, otherwise fallback to CPU ('cpu')
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)


# ------------------------------------------------------------
# ImageNet normalization because ResNet18 is pretrained
# ------------------------------------------------------------

# Define training image augmentation and normalization pipeline
train_transform = transforms.Compose([

    # Resize image dimensions to uniform spatial resolution
    transforms.Resize((224, 224)),

    # Randomly flip image horizontally for data augmentation
    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    # Randomly rotate image slightly to improve model generalization
    transforms.RandomRotation(
        degrees=10
    ),

    # Convert PIL Image to PyTorch FloatTensor with pixels scaled to [0.0, 1.0]
    transforms.ToTensor(),

    # Normalize image tensor using ImageNet standard mean and standard deviation
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Define testing image resizing and normalization pipeline
test_transform = transforms.Compose([

    # Resize image dimensions to uniform spatial resolution
    transforms.Resize((224, 224)),

    # Convert PIL Image to PyTorch FloatTensor with pixels scaled to [0.0, 1.0]
    transforms.ToTensor(),

    # Normalize image tensor using ImageNet standard mean and standard deviation
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


Device: cpu


In [ ]:
# ============================================================
#  NORMAL DATASET
# ============================================================

# Custom PyTorch Dataset class for loading and preprocessing Glaucoma fundus images
class GlaucomaDataset(Dataset):

    def __init__(
        self,
        image_paths,
        labels,
        transform=None
    ):

        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform


    # Return total number of samples present in the dataset
    def __len__(self):

        return len(
            self.image_paths
        )


    # Retrieve and transform the sample image and target label at given index
    def __getitem__(self, index):

        image_path = self.image_paths[index]

        label = self.labels[index]


        # ----------------------------------------------------
        # Load image
        # ----------------------------------------------------

        # Open image file from disk and convert to 3-channel RGB color format
        image = Image.open(
            image_path
        ).convert("RGB")


        # ----------------------------------------------------
        # Apply transform
        # ----------------------------------------------------

        # Apply transformation pipeline if specified
        if self.transform is not None:

            # Transform raw image into normalized PyTorch tensor
            image = self.transform(
                image
            )


        # Convert integer class label into PyTorch LongTensor
        label = torch.tensor(
            label,
            dtype=torch.long
        )


        # Return processed image tensor and corresponding class label
        return image, label


# ============================================================
# TEST DATASET
# ============================================================

# Instantiate GlaucomaDataset object for the test set
test_dataset = GlaucomaDataset(

    test_paths,

    test_labels,

    transform=test_transform
)


# Create DataLoader for test set to handle batching
test_loader = DataLoader(

    test_dataset,

    batch_size=16,

    shuffle=False,

    num_workers=0
)


print(
    "\nTest dataset created:",
    len(test_dataset)
)


Test dataset created: 18


In [ ]:

#  WEIGHTED DATASET FOR ADABOOST

# Custom Dataset class tracking sample weights for AdaBoost training
class WeightedGlaucomaDataset(Dataset):

    # Constructor method to initialize dataset paths, labels, and transforms
    def __init__(self,image_paths,labels,sample_weights,transform=None):

        self.image_paths = image_paths
        self.labels = labels
        self.sample_weights = sample_weights
        self.transform = transform

    # Return total number of samples present in the dataset
    def __len__(self):

        return len(
            self.image_paths
        )

    # Retrieve and transform the sample image and target label at given index
    def __getitem__(self, index):

        image_path = self.image_paths[index]

        label = self.labels[index]

        weight = self.sample_weights[index]


        # ----------------------------------------------------
        # Load image
        # ----------------------------------------------------

        # Open image file from disk and convert to 3-channel RGB color format
        image = Image.open(
            image_path
        ).convert("RGB")


        # ----------------------------------------------------
        # Transform
        # ----------------------------------------------------

        # Apply transformation pipeline if specified
        if self.transform is not None:

            # Transform raw image into normalized PyTorch tensor
            image = self.transform(
                image
            )


        # Convert integer class label into PyTorch LongTensor
        label = torch.tensor(
            label,
            dtype=torch.long
        )

        weight = torch.tensor(
            weight,
            dtype=torch.float32
        )


        return (
            image,
            label,
            weight,
            index
        )

In [ ]:
# ============================================================
#  CREATE RESNET18
# ============================================================

# Function to construct ResNet-18 architecture with pretrained weights
def create_resnet18():

    # Load pretrained ResNet-18 model architecture
    model = models.resnet18(
        weights=models.ResNet18_Weights.DEFAULT
    )

    # Retrieve number of input features entering the final classification layer
    num_features = model.fc.in_features

    # Binary classification
    # Class 0 = Normal
    # Class 1 = Glaucoma

    # Replace final fully connected layer with a Linear layer outputting 2 classes
    model.fc = nn.Linear(
        num_features,
        2
    )

    # Return the configured neural network model
    return model

In [ ]:
# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_boosting_model(model,dataloader,optimizer,epochs,model_name):

    print("\n")
    print("=" * 70)
    print("TRAINING:", model_name)
    print("=" * 70)


    for epoch in range(epochs):

        # Put neural network model in training mode (enables dropout and batchnorm updates)
        model.train()

        running_loss = 0.0

        correct = 0

        total = 0


        for (images,labels,weights,indices) in dataloader:


            images = images.to(device)

            labels = labels.to(device)

            weights = weights.to(device)


            # ------------------------------------------------
            # Clear gradients
            # ------------------------------------------------

            # Clear previously accumulated parameter gradients before current backward pass
            optimizer.zero_grad()


            # ------------------------------------------------
            # Forward pass
            # ------------------------------------------------

            # Execute forward pass: feed batch images into the neural network to get logits
            outputs = model(images)

            # ------------------------------------------------
            # Individual loss
            # ------------------------------------------------
            individual_loss = nn.functional.cross_entropy(outputs,labels,reduction="none")
            # ------------------------------------------------
            # Proper weighted mean loss
            # ------------------------------------------------
            weighted_loss = (individual_loss * weights).sum() / weights.sum().clamp_min(1e-8)

            # ------------------------------------------------
            # Backpropagation
            # ------------------------------------------------
            weighted_loss.backward()

            # ------------------------------------------------
            # Update model
            # ------------------------------------------------
            optimizer.step()

            # ------------------------------------------------
            # Statistics
            # ------------------------------------------------

            batch_size = images.size(0)

            running_loss += (
                weighted_loss.item()
                * batch_size
            )

            predicted = torch.argmax(
                outputs,
                dim=1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()


        epoch_loss = (
            running_loss / total
        )

        epoch_accuracy = (
            correct / total
        )


        print(

            f"Epoch [{epoch + 1}/{epochs}] "

            f"Loss: {epoch_loss:.4f} "

            f"Accuracy: {epoch_accuracy:.4f}"

        )

In [ ]:
# ============================================================
#  TRAINING-SET PREDICTION FUNCTION
# ============================================================

def get_train_predictions(model,dataloader):

    # Put neural network model in evaluation mode (disables dropout and fixes batchnorm)
    model.eval()


    number_of_samples = len(dataloader.dataset)

    predictions = np.zeros(number_of_samples,dtype=np.int64)

    probabilities = np.zeros(number_of_samples,dtype=np.float32)


    # Disable gradient calculations to accelerate inference and conserve GPU memory
    with torch.no_grad():

        for (
            images,
            labels,
            weights,
            indices
        ) in dataloader:


            images = images.to(device)


            outputs = model(
                images
            )


            probabilities_batch = torch.softmax(
                outputs,
                dim=1
            )[:, 1]


            predictions_batch = torch.argmax(
                outputs,
                dim=1
            )


            indices_np = (
                indices.cpu().numpy()
            )


            predictions[
                indices_np
            ] = predictions_batch.cpu().numpy()


            probabilities[
                indices_np
            ] = probabilities_batch.cpu().numpy()


    return (predictions,probabilities)

In [ ]:

#  TEST PREDICTION FUNCTION

def get_test_predictions(model,dataloader):

    # Put neural network model in evaluation mode (disables dropout and fixes batchnorm)
    model.eval()

    predictions = []

    probabilities = []


    # Disable gradient calculations to accelerate inference and conserve GPU memory
    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device)


            outputs = model(
                images
            )


            # Apply softmax function to logits to obtain normalized class probabilities
            probs = torch.softmax(
                outputs,
                dim=1
            )


            predicted = torch.argmax(
                outputs,
                dim=1
            )


            predictions.extend(
                predicted.cpu().numpy()
            )


            probabilities.extend(
                probs[:, 1].cpu().numpy()
            )


    return (

        np.array(
            predictions,
            dtype=np.int64
        ),

        np.array(
            probabilities,
            dtype=np.float32
        )

    )

In [ ]:

# INITIALIZE ADABOOST SAMPLE WEIGHTS

number_of_training_samples = len(train_paths)

# Initially all samples have equal importance

sample_weights = (np.ones(number_of_training_samples,dtype=np.float64)/ number_of_training_samples)


print("\n" + "=" * 70)
print("INITIAL SAMPLE WEIGHTS")
print("=" * 70)

print("Number of training samples:",number_of_training_samples)

print("Initial weight per sample:",sample_weights[0])

print(
    "Sum of weights:",
    sample_weights.sum()
)


INITIAL SAMPLE WEIGHTS
Number of training samples: 68
Initial weight per sample: 0.014705882352941176
Sum of weights: 0.9999999999999999


In [ ]:

#  ADABOOST TRAINING

NUM_MODELS = 3

NUM_EPOCHS = 5

# ------------------------------------------------------------
# These two lists MUST be created before the loop
# ------------------------------------------------------------

models_list = []

alpha_list = []


print("\n" + "=" * 70)
print("STARTING ADABOOST TRAINING")
print("=" * 70)


# BOOSTING LOOP

for model_number in range(NUM_MODELS):


    print("\n")
    print("#" * 70)

    print(f"BOOSTING ROUND " f"{model_number + 1}/{NUM_MODELS}")

    print("#" * 70)

    # ========================================================
    # CREATE WEIGHTED DATASET
    # ========================================================

    weighted_dataset = WeightedGlaucomaDataset(train_paths,train_labels,sample_weights,transform=train_transform)

    weighted_loader = DataLoader( weighted_dataset,batch_size=16,shuffle=True,num_workers=0)

    # CREATE NEW RESNET18

    model = create_resnet18()

    model = model.to(device)


    # ========================================================
    # OPTIMIZER
    # ========================================================

    # Initialize Adam optimizer with model weights and learning rate
    optimizer = torch.optim.Adam(model.parameters(),lr=0.0001,betas=(0.9, 0.999))

    # TRAIN CURRENT CNN

    train_boosting_model(model,weighted_loader,optimizer, NUM_EPOCHS, f"ResNet18 - Boosting Model "
        f"{model_number + 1}"

    )

    # ========================================================
    # GET TRAINING PREDICTIONS
    # ========================================================

    train_predictions, train_probabilities = (get_train_predictions( model,weighted_loader))

    # IDENTIFY INCORRECT PREDICTIONS
    incorrect = (train_predictions!=train_labels)

    # ========================================================
    # CALCULATE WEIGHTED ERROR
    # ========================================================

    weighted_error = np.sum(sample_weights*incorrect.astype(np.float64))


    print(
        "\nWeighted Error:",
        weighted_error
    )


    # ========================================================
    # CHECK WHETHER MODEL IS BETTER THAN RANDOM
    # ========================================================

    if weighted_error >= 0.5:

        print(
            "\nWARNING:"
        )

        print(
            "Weighted error is >= 0.5."
        )

        print(
            "This model is not suitable as a "
            "positive AdaBoost learner."
        )

        # Give zero influence to this model
        alpha = 0.0

    else:

        # ====================================================
        # NUMERICAL SAFETY
        # ====================================================

        weighted_error = np.clip(weighted_error,1e-10,1.0 - 1e-10)

        # ====================================================
        # CALCULATE ALPHA
        # ====================================================
        alpha = (0.5*np.log((1.0 - weighted_error)/weighted_error))


    print(
        "CNN Model Weight (alpha):",alpha)


    # ========================================================
    # UPDATE SAMPLE WEIGHTS
    # ========================================================

    if alpha != 0.0:

        sample_weights *= np.exp(alpha*incorrect.astype(np.float64))


        # ----------------------------------------------------
        # Normalize
        # ----------------------------------------------------

        sample_weights /= (sample_weights.sum())

    # ========================================================
    # PRINT UPDATED WEIGHTS
    # ========================================================

    print(
        "\nMinimum sample weight:",
        sample_weights.min()
    )

    print(
        "Maximum sample weight:",
        sample_weights.max()
    )

    print(
        "Sum of sample weights:",
        sample_weights.sum()
    )

    # STORE MODEL

    models_list.append(model)

    # STORE ALPHA

    alpha_list.append(alpha)

    print(
        f"\nModel {model_number + 1} stored successfully."
    )

    print(
        "Total models stored:",
        len(models_list)
    )

    print(
        "Total alpha values:",
        len(alpha_list)
    )

# PRINT FINAL MODEL WEIGHTS

print("\n" + "=" * 70)
print("BOOSTING MODEL WEIGHTS")
print("=" * 70)


for i, alpha in enumerate(alpha_list):

    print(

        f"Model {i + 1} "
        f"(ResNet18) "
        f"alpha = {alpha:.6f}"

    )

print("\nTraining completed.")

print(
    "Number of models:",
    len(models_list)
)


STARTING ADABOOST TRAINING


######################################################################
BOOSTING ROUND 1/3
######################################################################


TRAINING: ResNet18 - Boosting Model 1
Epoch [1/5] Loss: 0.7649 Accuracy: 0.5000
Epoch [2/5] Loss: 0.5241 Accuracy: 0.8088
Epoch [3/5] Loss: 0.5124 Accuracy: 0.7647
Epoch [4/5] Loss: 0.4298 Accuracy: 0.8382
Epoch [5/5] Loss: 0.3446 Accuracy: 0.8971

Weighted Error: 0.044117647058823525
CNN Model Weight (alpha): 1.5378874906137638

Minimum sample weight: 0.01266396028690597
Maximum sample weight: 0.05894752711703734
Sum of sample weights: 1.0000000000000002

Model 1 stored successfully.
Total models stored: 1
Total alpha values: 1


######################################################################
BOOSTING ROUND 2/3
######################################################################


TRAINING: ResNet18 - Boosting Model 2
Epoch [1/5] Loss: 0.7187 Accuracy: 0.5147
Epoch [2/5] Loss: 0.5793 Ac

In [ ]:
# ============================================================
#  VERIFY BOOSTING MODELS
# ============================================================

print("\n" + "=" * 70)
print("MODEL VERIFICATION")
print("=" * 70)

print(
    "Expected models:",
    NUM_MODELS
)

print(
    "Stored models:",
    len(models_list)
)

print(
    "Stored alpha values:",
    len(alpha_list)
)


if len(models_list) != NUM_MODELS:

    raise RuntimeError(

        f"Expected {NUM_MODELS} models, "
        f"but found {len(models_list)}."

    )


if len(alpha_list) != NUM_MODELS:

    raise RuntimeError(

        f"Expected {NUM_MODELS} alpha values, "
        f"but found {len(alpha_list)}."

    )


print("\nAll boosting models are available.")


MODEL VERIFICATION
Expected models: 3
Stored models: 3
Stored alpha values: 3

All boosting models are available.


In [ ]:
# ============================================================
# TEST PREDICTIONS FROM ALL 3 CNNs
# ============================================================

test_predictions = []

test_probabilities = []


print("\n" + "=" * 70)
print("GENERATING TEST PREDICTIONS")
print("=" * 70)


for model_index, model in enumerate(
    models_list
):


    print(
        f"\nTesting Model "
        f"{model_index + 1}/{NUM_MODELS}"
    )


    predictions, probabilities = get_test_predictions(model,test_loader)


    test_predictions.append(
        predictions
    )


    test_probabilities.append(
        probabilities
    )


    print(
        f"Model {model_index + 1}: "
        f"{len(predictions)} predictions"
    )


# ============================================================
# CONVERT TO NUMPY ARRAYS
# ============================================================

test_predictions = np.array(
    test_predictions
)

test_probabilities = np.array(
    test_probabilities
)


# ============================================================
# CHECK SHAPE
# ============================================================

print("\n" + "=" * 70)
print("TEST PREDICTION SHAPE")
print("=" * 70)

print(
    "test_predictions shape:",
    test_predictions.shape
)

print(
    "test_probabilities shape:",
    test_probabilities.shape
)

print(
    "test_labels shape:",
    test_labels.shape
)


# ============================================================
# SAFETY CHECK
# ============================================================

if test_predictions.shape[0] != NUM_MODELS:

    raise RuntimeError(

        f"Expected predictions from "
        f"{NUM_MODELS} models, but found "
        f"{test_predictions.shape[0]}."

    )


if test_predictions.shape[1] != len(test_labels):

    raise RuntimeError(

        "Number of test predictions does not "
        "match number of test labels."

    )


print(
    "\nPrediction arrays are correct."
)


GENERATING TEST PREDICTIONS

Testing Model 1/3
Model 1: 18 predictions

Testing Model 2/3
Model 2: 18 predictions

Testing Model 3/3
Model 3: 18 predictions

TEST PREDICTION SHAPE
test_predictions shape: (3, 18)
test_probabilities shape: (3, 18)
test_labels shape: (18,)

Prediction arrays are correct.


In [ ]:
# ============================================================
#  ADABOOST WEIGHTED VOTING
# ============================================================

print("\n" + "=" * 70)
print("ADABOOST WEIGHTED VOTING")
print("=" * 70)


# ------------------------------------------------------------
# Initialize final score
# ------------------------------------------------------------

final_scores = np.zeros(len(test_labels),dtype=np.float64)


# ============================================================
# COMBINE ALL CNN PREDICTIONS
# ============================================================

for model_index in range(NUM_MODELS):


    # --------------------------------------------------------
    # Current CNN predictions
    # --------------------------------------------------------

    prediction = test_predictions[model_index]

    # --------------------------------------------------------
    # Convert:
    #
    # class 0 → -1
    # class 1 → +1
    # --------------------------------------------------------

    signed_prediction = (2 * prediction-1)


    # --------------------------------------------------------
    # Weighted contribution
    # --------------------------------------------------------

    contribution = (

        alpha_list[model_index]
        *
        signed_prediction

    )


    # --------------------------------------------------------
    # Add to final score
    # --------------------------------------------------------

    final_scores += contribution


    print(

        f"Model {model_index + 1}: "
        f"alpha = "
        f"{alpha_list[model_index]:.6f}"

    )


# ============================================================
# FINAL CLASSIFICATION
# ============================================================

final_predictions = (

    final_scores > 0

).astype(np.int64)


print(
    "\nFinal predictions generated."
)


ADABOOST WEIGHTED VOTING
Model 1: alpha = 1.537887
Model 2: alpha = 1.047399
Model 3: alpha = 1.265586

Final predictions generated.


In [ ]:
# ============================================================
#  FINAL BOOSTING PROBABILITY
# ============================================================

# ------------------------------------------------------------
# Sigmoid conversion
# ------------------------------------------------------------

# Clip scores to prevent numerical overflow

safe_scores = np.clip(
    final_scores,
    -50,
    50
)


final_probabilities = (

    1.0
    /
    (
        1.0
        +
        np.exp(-safe_scores)
    )

)


print("\n" + "=" * 70)
print("FINAL BOOSTING OUTPUT")
print("=" * 70)

print(
    "Final prediction shape:",
    final_predictions.shape
)

print(
    "Final probability shape:",
    final_probabilities.shape
)


FINAL BOOSTING OUTPUT
Final prediction shape: (18,)
Final probability shape: (18,)


In [ ]:
# ============================================================
#  METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
    y_prob,
    model_name
):


    # Compute overall classification accuracy (correct predictions / total)
    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    # Compute confusion matrix detailing TN, FP, FN, and TP counts
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )


    tn, fp, fn, tp = cm.ravel()


    # --------------------------------------------------------
    # Sensitivity / Recall
    # --------------------------------------------------------

    sensitivity = (

        tp / (tp + fn)

        if (tp + fn) > 0

        else 0.0

    )


    # --------------------------------------------------------
    # Specificity
    # --------------------------------------------------------

    specificity = (

        tn / (tn + fp)

        if (tn + fp) > 0

        else 0.0

    )


    # --------------------------------------------------------
    # Precision
    # --------------------------------------------------------

    # Compute precision score: TP / (TP + FP)
    precision = precision_score(

        y_true,

        y_pred,

        zero_division=0

    )


    # --------------------------------------------------------
    # F1
    # --------------------------------------------------------

    # Compute F1 score: harmonic mean of precision and recall
    f1 = f1_score(

        y_true,

        y_pred,

        zero_division=0

    )


    # --------------------------------------------------------
    # AUC
    # --------------------------------------------------------

    try:

        # Compute Area Under the Receiver Operating Characteristic curve (ROC-AUC)
        auc = roc_auc_score(

            y_true,

            y_prob

        )

    except ValueError:

        auc = np.nan


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("\n" + "=" * 70)

    print(
        model_name
    )

    print("=" * 70)


    print("\nConfusion Matrix:")

    print(cm)


    print(
        f"\nAccuracy     : {accuracy:.4f}"
    )

    print(
        f"Sensitivity  : {sensitivity:.4f}"
    )

    print(
        f"Specificity  : {specificity:.4f}"
    )

    print(
        f"Precision     : {precision:.4f}"
    )

    print(
        f"F1-score      : {f1:.4f}"
    )

    print(
        f"AUC           : {auc:.4f}"
    )


    return {

        "Model": model_name,

        "Accuracy": accuracy,

        "Sensitivity": sensitivity,

        "Specificity": specificity,

        "Precision": precision,

        "F1": f1,

        "AUC": auc

    }

In [ ]:
# ============================================================
#  EVALUATE INDIVIDUAL CNNs
# ============================================================

individual_results = []


for i in range(NUM_MODELS):


    result = calculate_metrics(test_labels,test_predictions[i],test_probabilities[i],f"Boosting CNN {i + 1} - ResNet18")

    individual_results.append(result)


Boosting CNN 1 - ResNet18

Confusion Matrix:
[[7 3]
 [2 6]]

Accuracy     : 0.7222
Sensitivity  : 0.7500
Specificity  : 0.7000
Precision     : 0.6667
F1-score      : 0.7059
AUC           : 0.6500

Boosting CNN 2 - ResNet18

Confusion Matrix:
[[8 2]
 [6 2]]

Accuracy     : 0.5556
Sensitivity  : 0.2500
Specificity  : 0.8000
Precision     : 0.5000
F1-score      : 0.3333
AUC           : 0.5750

Boosting CNN 3 - ResNet18

Confusion Matrix:
[[5 5]
 [4 4]]

Accuracy     : 0.5000
Sensitivity  : 0.5000
Specificity  : 0.5000
Precision     : 0.4444
F1-score      : 0.4706
AUC           : 0.5375


In [ ]:
# ============================================================
#  EVALUATE FINAL ADABOOST MODEL
# ============================================================

boosting_result = calculate_metrics(test_labels,final_predictions,final_probabilities,"FINAL ADABOOST BOOSTING MODEL")


FINAL ADABOOST BOOSTING MODEL

Confusion Matrix:
[[7 3]
 [4 4]]

Accuracy     : 0.6111
Sensitivity  : 0.5000
Specificity  : 0.7000
Precision     : 0.5714
F1-score      : 0.5333
AUC           : 0.6813


In [ ]:
# ============================================================
#  PERFORMANCE COMPARISON
# ============================================================

results_df = pd.DataFrame(

    individual_results
    +
    [boosting_result]

)


print("\n" + "=" * 70)

print(
    "PERFORMANCE COMPARISON"
)

print("=" * 70)


print(
    results_df.to_string(
        index=False
    )
)


PERFORMANCE COMPARISON
                        Model  Accuracy  Sensitivity  Specificity  Precision       F1     AUC
    Boosting CNN 1 - ResNet18  0.722222         0.75          0.7   0.666667 0.705882 0.65000
    Boosting CNN 2 - ResNet18  0.555556         0.25          0.8   0.500000 0.333333 0.57500
    Boosting CNN 3 - ResNet18  0.500000         0.50          0.5   0.444444 0.470588 0.53750
FINAL ADABOOST BOOSTING MODEL  0.611111         0.50          0.7   0.571429 0.533333 0.68125


In [ ]:
# ============================================================
#  CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)

print(
    "FINAL ADABOOST CLASSIFICATION REPORT"
)

print("=" * 70)


print(

    classification_report(

        test_labels,

        final_predictions,

        labels=[0, 1],

        target_names=[

            str(index_to_label[0]),

            str(index_to_label[1])

        ],

        zero_division=0

    )
)


FINAL ADABOOST CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.64      0.70      0.67        10
           1       0.57      0.50      0.53         8

    accuracy                           0.61        18
   macro avg       0.60      0.60      0.60        18
weighted avg       0.61      0.61      0.61        18



In [ ]:
# ============================================================
#  SAVE TRAINED CNN MODELS
# ============================================================

print("\n" + "=" * 70)

print(
    "SAVING TRAINED MODELS"
)

print("=" * 70)


for i, model in enumerate(
    models_list
):


    model_path = os.path.join(

        path2data,

        f"Boosting_ResNet18_Model_{i + 1}.pth"

    )


    # Serialize and save model weights state dictionary checkpoint to disk
    torch.save(

        model.state_dict(),

        model_path

    )


    print(
        "Saved:",
        model_path
    )


SAVING TRAINED MODELS
Saved: F:/Amrita_2024/DEEP LEARNING FOR SIGNAL & IMAGE _21AIE312/Python_neural network/Boosting_ResNet18_Model_1.pth
Saved: F:/Amrita_2024/DEEP LEARNING FOR SIGNAL & IMAGE _21AIE312/Python_neural network/Boosting_ResNet18_Model_2.pth
Saved: F:/Amrita_2024/DEEP LEARNING FOR SIGNAL & IMAGE _21AIE312/Python_neural network/Boosting_ResNet18_Model_3.pth


In [ ]:
# ============================================================
#  SAVE PREDICTIONS TO EXCEL
# ============================================================

prediction_results = pd.DataFrame({

    "Image": test_paths,


    "True_Label": [

        index_to_label[int(x)]

        for x in test_labels

    ],


    "CNN1_ResNet18": [

        index_to_label[int(x)]

        for x in test_predictions[0]

    ],


    "CNN2_ResNet18": [

        index_to_label[int(x)]

        for x in test_predictions[1]

    ],


    "CNN3_ResNet18": [

        index_to_label[int(x)]

        for x in test_predictions[2]

    ],


    "Boosting_Prediction": [

        index_to_label[int(x)]

        for x in final_predictions

    ],


    "CNN1_Probability": (
        test_probabilities[0]
    ),


    "CNN2_Probability": (
        test_probabilities[1]
    ),


    "CNN3_Probability": (
        test_probabilities[2]
    ),


    "Boosting_Score": (
        final_scores
    ),


    "Boosting_Probability": (
        final_probabilities
    )

})


output_file = os.path.join(

    path2data,

    "3_CNN_AdaBoost_Glaucoma_Results.xlsx"

)


prediction_results.to_excel(

    output_file,

    index=False

)


print("\nPrediction results saved to:")

print(
    output_file
)


Prediction results saved to:
F:/Amrita_2024/DEEP LEARNING FOR SIGNAL & IMAGE _21AIE312/Python_neural network/3_CNN_AdaBoost_Glaucoma_Results.xlsx


In [ ]:
# ============================================================
#  FINAL SUMMARY
# ============================================================

print("\n")

print("=" * 70)

print(
    "FINAL ADABOOST SUMMARY"
)

print("=" * 70)


print(
    "Dataset              : DRION Glaucoma"
)

print(
    "CNN architecture     : ResNet18"
)

print(
    "Number of CNN models : 3"
)

print(
    "Boosting method      : AdaBoost-style"
)

print(
    "Training             : Sequential"
)

print(
    "Sample weighting     : Adaptive"
)

print(
    "Fusion               : Alpha-weighted Voting"
)


print(
    f"\nFinal Accuracy       : "
    f"{boosting_result['Accuracy']:.4f}"
)


print(
    f"Final Sensitivity    : "
    f"{boosting_result['Sensitivity']:.4f}"
)


print(
    f"Final Specificity    : "
    f"{boosting_result['Specificity']:.4f}"
)


print(
    f"Final Precision      : "
    f"{boosting_result['Precision']:.4f}"
)


print(
    f"Final F1-score       : "
    f"{boosting_result['F1']:.4f}"
)


print(
    f"Final AUC            : "
    f"{boosting_result['AUC']:.4f}"
)


print("\nModel alphas:")

for i, alpha in enumerate(alpha_list):

    print(
        f"  CNN {i + 1}: "
        f"{alpha:.6f}"
    )


print("\n" + "=" * 70)

print(
    "ADABOOST TRAINING AND EVALUATION COMPLETED"
)

print("=" * 70)



FINAL ADABOOST SUMMARY
Dataset              : DRION Glaucoma
CNN architecture     : ResNet18
Number of CNN models : 3
Boosting method      : AdaBoost-style
Training             : Sequential
Sample weighting     : Adaptive
Fusion               : Alpha-weighted Voting

Final Accuracy       : 0.6111
Final Sensitivity    : 0.5000
Final Specificity    : 0.7000
Final Precision      : 0.5714
Final F1-score       : 0.5333
Final AUC            : 0.6813

Model alphas:
  CNN 1: 1.537887
  CNN 2: 1.047399
  CNN 3: 1.265586

ADABOOST TRAINING AND EVALUATION COMPLETED
